# Travel Reimbursement Approval Agent

This notebook checks the five travel claims from the assignment against the supplied reimbursement policy.

It checks expenses, receipts, limits, approval levels and submission timing.

The final section gives one result for each claim.

In [ ]:
pip install pandas matplotlib openai


## Policy

The rules below are from Appendix A of the assignment.

| Rule | Description |
|---|---|
| POL-CAT-01 | Economy airfare, lodging, meals, ground transport and conference fees are eligible |
| POL-CAT-02 | Spa, minibar, personal expenses, fines and similar items are not eligible |
| POL-PD-01 | Meals: $75 per day |
| POL-PD-02 | Lodging: $200 per night |
| POL-PD-03 | Ground transport: $50 per day |
| POL-AIR-01 | Business/first class airfare goes to manual review |
| POL-RCT-01 | Receipts are required over $25; airfare and lodging always require one |
| POL-RCT-02 | Missing required receipt goes to manual review |
| POL-APR-01 | Up to $500 |
| POL-APR-02 | Over $500 up to $2,000 |
| POL-APR-03 | Over $2,000 goes to manual review |
| POL-TIME-01 | Claim must be submitted within 30 days |

In [ ]:

import os
import json
from copy import deepcopy
from datetime import date
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

pd.set_option("display.max_colwidth", 100)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

USE_LLM = bool(OPENAI_API_KEY and OpenAI is not None)

print(f"LLM tool-calling enabled: {USE_LLM}")
if not USE_LLM:
    print("Running with deterministic fallback agent. Set OPENAI_API_KEY to enable live GenAI orchestration.")


## 2. Input Claims

The following five claims are the exact sample claims supplied in Appendix B.

In [ ]:

CLAIMS = [
    {
        "claim_id": "CLM-001",
        "employee": "A. Rivera",
        "trip_start": "2026-06-10",
        "trip_end": "2026-06-12",
        "submitted": "2026-06-20",
        "items": [
            {"category": "airfare", "description": "Round-trip economy airfare", "amount": 420.00, "receipt_attached": True},
            {"category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": 360.00, "receipt_attached": True, "nights": 2},
            {"category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": 180.00, "receipt_attached": True, "days": 3},
            {"category": "conference_fees", "description": "Conference registration", "amount": 150.00, "receipt_attached": True},
        ],
    },
    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "trip_start": "2026-06-14",
        "trip_end": "2026-06-15",
        "submitted": "2026-06-25",
        "items": [
            {"category": "spa", "description": "Hotel spa package", "amount": 300.00, "receipt_attached": True},
            {"category": "minibar", "description": "In-room minibar", "amount": 80.00, "receipt_attached": True},
        ],
    },
    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "trip_start": "2026-06-08",
        "trip_end": "2026-06-10",
        "submitted": "2026-06-22",
        "items": [
            {"category": "airfare", "description": "Round-trip economy airfare", "amount": 300.00, "receipt_attached": True},
            {"category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": 500.00, "receipt_attached": True, "nights": 2},
            {"category": "meals", "description": "Meals, 2 days @ $70/day", "amount": 140.00, "receipt_attached": True, "days": 2},
        ],
    },
    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "trip_start": "2026-06-16",
        "trip_end": "2026-06-18",
        "submitted": "2026-06-28",
        "items": [
            {"category": "airfare", "description": "Business-class international airfare", "amount": 2400.00, "receipt_attached": True, "airfare_class": "business"},
            {"category": "lodging", "description": "Hotel, 3 nights", "amount": 600.00, "receipt_attached": False, "nights": 3},
        ],
    },
    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "trip_start": "2026-06-11",
        "trip_end": "2026-06-11",
        "submitted": "2026-06-24",
        "items": [
            {"category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.00, "receipt_attached": False, "days": 1},
        ],
    },
]

assert len(CLAIMS) == 5
print(json.dumps(CLAIMS, indent=2))


## 3. Tool Layer

Each tool has a narrow responsibility and returns structured evidence. The policy decision layer consumes tool outputs rather than relying on free-form model memory.

In [ ]:

POLICY = {
    "POL-CAT-01": {
        "text": "Eligible categories: economy airfare, lodging, meals, ground transport, conference/registration.",
        "eligible": {"airfare", "lodging", "meals", "ground_transport", "conference_fees"},
    },
    "POL-CAT-02": {
        "text": "Ineligible: alcohol/minibar, spa/gym/entertainment, personal shopping/gifts, fines/penalties/late fees, personal expenses.",
        "ineligible": {"alcohol", "minibar", "spa", "gym", "entertainment", "personal_shopping", "gifts", "traffic_fine", "penalty", "late_fee", "personal"},
    },
    "POL-PD-01": {"text": "Meals maximum $75 per day.", "category": "meals", "cap": 75.00},
    "POL-PD-02": {"text": "Lodging maximum $200 per night.", "category": "lodging", "cap": 200.00},
    "POL-PD-03": {"text": "Ground transport maximum $50 per day.", "category": "ground_transport", "cap": 50.00},
    "POL-AIR-01": {"text": "Only economy airfare is reimbursable. Business/first class is a policy exception requiring Manual Review."},
    "POL-RCT-01": {"text": "Receipt required above $25; airfare and lodging always require a receipt."},
    "POL-RCT-02": {"text": "Missing required receipt routes the item/claim to Manual Review."},
    "POL-APR-01": {"text": "Total <= $500 may be auto-approved if fully compliant.", "max": 500.00},
    "POL-APR-02": {"text": "Total > $500 and <= $2,000 is approvable if fully compliant.", "max": 2000.00},
    "POL-APR-03": {"text": "Total > $2,000 exceeds agent auto-approval authority and requires Manual Review."},
    "POL-TIME-01": {"text": "Claims must be submitted within 30 days of the expense date; late claims route to Manual Review."},
}

def policy_lookup(claim):
    """Return the policy rules needed for this claim."""
    refs = {"POL-CAT-01", "POL-CAT-02", "POL-RCT-01", "POL-RCT-02", "POL-TIME-01"}
    categories = {i["category"] for i in claim["items"]}
    if "meals" in categories:
        refs.add("POL-PD-01")
    if "lodging" in categories:
        refs.add("POL-PD-02")
    if "ground_transport" in categories:
        refs.add("POL-PD-03")
    if any(i["category"] == "airfare" for i in claim["items"]):
        refs.add("POL-AIR-01")
    return {
        "tool": "policy_lookup",
        "policy_refs": sorted(refs),
        "policy_text": {r: POLICY[r]["text"] for r in sorted(refs)}
    }

def receipt_completeness_check(claim):
    """Check receipt requirements for each item."""
    missing = []
    checked = []
    for item in claim["items"]:
        required = item["category"] in {"airfare", "lodging"} or item["amount"] > 25
        checked.append({
            "category": item["category"],
            "amount": item["amount"],
            "required": required,
            "attached": bool(item.get("receipt_attached", False))
        })
        if required and not item.get("receipt_attached", False):
            missing.append(item["category"])
    return {
        "tool": "receipt_completeness_check",
        "required_receipts_present": len(missing) == 0,
        "missing_docs": sorted(set(missing)),
        "checked_items": checked
    }

def limit_checker(claim):
    """Apply the category limits."""
    results = []
    approved = 0.0
    deducted = 0.0
    cap_refs = []

    for item in claim["items"]:
        amount = float(item["amount"])
        category = item["category"]

        if category == "meals":
            cap = 75.0 * float(item.get("days", 1))
            ref = "POL-PD-01"
        elif category == "lodging":
            cap = 200.0 * float(item.get("nights", 1))
            ref = "POL-PD-02"
        elif category == "ground_transport":
            cap = 50.0 * float(item.get("days", 1))
            ref = "POL-PD-03"
        else:
            cap = amount
            ref = None

        if category in POLICY["POL-CAT-02"]["ineligible"]:
            reimbursable = 0.0
            item_deducted = amount
        else:
            reimbursable = min(amount, cap)
            item_deducted = amount - reimbursable

        if ref and amount > cap:
            cap_refs.append(ref)

        approved += reimbursable
        deducted += item_deducted
        results.append({
            "category": category,
            "claimed": round(amount, 2),
            "cap": round(cap, 2),
            "reimbursable_before_manual_review": round(reimbursable, 2),
            "deducted_by_category_rule": round(item_deducted, 2),
        })

    return {
        "tool": "limit_checker",
        "items": results,
        "approved_before_manual_review": round(approved, 2),
        "deducted_before_manual_review": round(deducted, 2),
        "policy_refs": sorted(set(cap_refs))
    }

def approval_threshold_check(reimbursable_amount):
    """Check the approval level for the amount."""
    amount = float(reimbursable_amount)
    if amount <= 500:
        tier = "AUTO_APPROVE"
        manual = False
        ref = "POL-APR-01"
    elif amount <= 2000:
        tier = "MANAGER"
        manual = False
        ref = "POL-APR-02"
    else:
        tier = "DIRECTOR_MANUAL_REVIEW"
        manual = True
        ref = "POL-APR-03"
    return {
        "tool": "approval_threshold_check",
        "amount": round(amount, 2),
        "tier": tier,
        "requires_manual_review": manual,
        "policy_ref": ref
    }


In [ ]:

TOOL_FUNCTIONS = {
    "policy_lookup": policy_lookup,
    "receipt_completeness_check": receipt_completeness_check,
    "limit_checker": limit_checker,
}

TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "policy_lookup",
            "description": "Retrieve the policy rules relevant to a reimbursement claim.",
            "parameters": {
                "type": "object",
                "properties": {"claim": {"type": "object"}},
                "required": ["claim"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "receipt_completeness_check",
            "description": "Check whether required receipts are attached.",
            "parameters": {
                "type": "object",
                "properties": {"claim": {"type": "object"}},
                "required": ["claim"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "limit_checker",
            "description": "Calculate category-cap deductions and reimbursable amounts.",
            "parameters": {
                "type": "object",
                "properties": {"claim": {"type": "object"}},
                "required": ["claim"]
            }
        }
    },
]



## 4. Agent Workflow

The agent follows this control flow:

1. **Intake** the structured claim.
2. Ask the LLM (when configured) to select relevant tools.
3. Execute selected tools.
4. Always run the approval-threshold check after calculating the post-cap amount.
5. Apply deterministic policy guardrails:
   - ineligible-only → `REJECT`
   - required receipt missing → `MANUAL_REVIEW`
   - business/first airfare → `MANUAL_REVIEW`
   - post-cap total > $2,000 → `MANUAL_REVIEW`
   - late submission → `MANUAL_REVIEW`
   - compliant claim with cap deductions → `PARTIAL_APPROVE`
   - otherwise → `APPROVE`
6. Validate the result schema.
7. Produce an audit record containing tools used, policy references, and explanation.

This deliberately separates **LLM orchestration** from **policy enforcement**. A model can select tools and explain evidence, but it cannot override an explicit policy guardrail.


In [ ]:

SYSTEM_PROMPT = """
You are a travel reimbursement policy agent. You must ground your reasoning in the supplied policy.
Use tools when their evidence is needed. Do not invent policy rules.
Important:
- Missing required receipts must be Manual Review, not rejection.
- Business/first-class airfare is a policy exception and must be Manual Review.
- Claims over $2,000 after applicable caps are Manual Review.
- Ineligible categories are not reimbursable.
- Meal/lodging/ground-transport caps may create a partial approval.
Return concise reasoning based only on tool evidence.
"""

def llm_select_and_run_tools(claim):
    """Use LLM tool calling if configured; otherwise use a deterministic routing policy."""
    if not USE_LLM:
        # Local checks.
        selected = ["policy_lookup", "receipt_completeness_check", "limit_checker"]
        evidence = {}
        for name in selected:
            evidence[name] = TOOL_FUNCTIONS[name](claim)
        return evidence, selected, "deterministic_fallback"

    client = OpenAI(api_key=OPENAI_API_KEY)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Evaluate this claim. Decide which tools you need and call them. Claim:\n" + json.dumps(claim)}
    ]

    selected = []
    evidence = {}

    for _ in range(3):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
            temperature=0
        )
        msg = response.choices[0].message
        tool_calls = msg.tool_calls or []

        if not tool_calls:
            break

        messages.append(msg)
        for call in tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = TOOL_FUNCTIONS[name](args["claim"])
            selected.append(name)
            evidence[name] = result
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

    # Make sure all required checks are available.
    for name in ["policy_lookup", "receipt_completeness_check", "limit_checker"]:
        if name not in evidence:
            evidence[name] = TOOL_FUNCTIONS[name](claim)
            selected.append(name)

    return evidence, list(dict.fromkeys(selected)), "llm_tool_calling"

def submission_age_days(claim):
    expense_date = date.fromisoformat(claim["trip_start"])
    submitted = date.fromisoformat(claim["submitted"])
    return (submitted - expense_date).days

def evaluate_policy(claim, evidence):
    """Deterministic policy guardrail and final decision engine."""
    items = claim["items"]
    limit = evidence["limit_checker"]
    receipts = evidence["receipt_completeness_check"]

    policy_refs = set(evidence["policy_lookup"]["policy_refs"])
    missing_docs = list(receipts["missing_docs"])
    age = submission_age_days(claim)

    # Policy exceptions.
    business_airfare = any(
        i["category"] == "airfare" and i.get("airfare_class", "economy").lower() in {"business", "first"}
        for i in items
    )

    ineligible_items = [
        i for i in items
        if i["category"] in POLICY["POL-CAT-02"]["ineligible"]
    ]

    threshold = approval_threshold_check(limit["approved_before_manual_review"])

    if age > 30:
        policy_refs.add("POL-TIME-01")

    if business_airfare:
        policy_refs.add("POL-AIR-01")

    if missing_docs:
        policy_refs.update({"POL-RCT-01", "POL-RCT-02"})

    # Nothing eligible.
    if ineligible_items and len(ineligible_items) == len(items):
        decision = "REJECT"
        approved = 0.0
        deducted = round(sum(i["amount"] for i in items), 2)
        explanation = "All claimed items are ineligible under POL-CAT-02; nothing is reimbursable."

    # Manual review takes priority.
    elif business_airfare or missing_docs or age > 30 or threshold["requires_manual_review"]:
        policy_refs.add(threshold["policy_ref"])
        decision = "MANUAL_REVIEW"
        approved = 0.0
        deducted = 0.0
        reasons = []
        if business_airfare:
            reasons.append("business/first-class airfare is a policy exception")
        if missing_docs:
            reasons.append("required receipt(s) are missing")
        if age > 30:
            reasons.append("claim was submitted outside the 30-day window")
        if threshold["requires_manual_review"]:
            reasons.append("post-cap reimbursable amount exceeds agent approval authority")
        explanation = "Manual review required because " + "; ".join(reasons) + "."

    else:
        policy_refs.add(threshold["policy_ref"])
        approved = round(limit["approved_before_manual_review"], 2)
        deducted = round(sum(i["amount"] for i in items) - approved, 2)
        has_cap_deduction = any(
            i["category"] in {"meals", "lodging", "ground_transport"} and
            float(i["amount"]) > float(next(
                (r["cap"] for r in limit["items"] if r["category"] == i["category"]),
                i["amount"]
            ))
            for i in items
        )

        if has_cap_deduction:
            decision = "PARTIAL_APPROVE"
            explanation = "Claim is eligible, but one or more category amounts exceed the applicable policy cap; the excess is deducted."
        else:
            decision = "APPROVE"
            explanation = "Claim is fully compliant with category eligibility, receipt, timeliness, limit, and approval-tier rules."

    # Confidence in the classification.
    confidence = 0.99 if decision != "MANUAL_REVIEW" else 0.98

    return {
        "claim_id": claim["claim_id"],
        "decision": decision,
        "approved_amount": round(approved, 2),
        "deducted_amount": round(deducted, 2),
        "missing_docs": missing_docs,
        "policy_refs": sorted(policy_refs),
        "confidence": confidence,
        "explanation": explanation,
    }

def validate_result(result):
    required = [
        "claim_id", "decision", "approved_amount", "deducted_amount",
        "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"
    ]
    assert list(result.keys()) == required, f"Schema mismatch: {list(result.keys())}"
    assert result["decision"] in {"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"}
    assert isinstance(result["missing_docs"], list)
    assert isinstance(result["policy_refs"], list)
    assert 0 <= float(result["confidence"]) <= 1
    assert isinstance(result["tools_used"], list)
    return True

def run_agent(claim):
    evidence, tools_used, mode = llm_select_and_run_tools(claim)
    result = evaluate_policy(claim, evidence)
    result["tools_used"] = tools_used + ["approval_threshold_check"]
    result["tools_used"] = list(dict.fromkeys(result["tools_used"]))
    validate_result(result)
    return result, evidence, mode

all_results = []
audit_records = []

for claim in CLAIMS:
    result, evidence, mode = run_agent(claim)
    all_results.append(result)
    audit_records.append({
        "claim_id": claim["claim_id"],
        "mode": mode,
        "tools_used": result["tools_used"],
        "policy_refs": result["policy_refs"],
        "evidence": evidence,
    })

print(json.dumps(all_results, indent=2))


## 5. Sample Outputs

At least three examples are shown below before the final all-claims result.

In [ ]:

sample_ids = ["CLM-001", "CLM-003", "CLM-004"]
for r in all_results:
    if r["claim_id"] in sample_ids:
        print(json.dumps(r, indent=2))
        print("-" * 80)


## Dashboard

The dashboard is derived directly from the actual agent outcomes; no values are hard-coded.

In [ ]:

results_df = pd.DataFrame(all_results)

decision_counts = results_df["decision"].value_counts().reindex(
    ["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"], fill_value=0
)

summary = pd.DataFrame({
    "Metric": ["Claims", "Approved amount", "Deducted amount", "Manual review claims"],
    "Value": [
        len(results_df),
        results_df["approved_amount"].sum(),
        results_df["deducted_amount"].sum(),
        int((results_df["decision"] == "MANUAL_REVIEW").sum())
    ]
})
display(summary)

plt.figure(figsize=(8, 4))
decision_counts.plot(kind="bar")
plt.title("Decision Breakdown")
plt.xlabel("Decision")
plt.ylabel("Number of Claims")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
results_df.set_index("claim_id")[["approved_amount", "deducted_amount"]].plot(kind="bar", ax=plt.gca())
plt.title("Approved vs Deducted Amount by Claim")
plt.xlabel("Claim")
plt.ylabel("USD")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()



## Design Notes & Reasoning

### Why this architecture?

A fully autonomous LLM is the wrong place to encode financial policy. The prototype therefore uses a **hybrid agent**:

- **LLM layer:** optional tool-calling planner that decides which evidence it needs and produces a reasoning trace.
- **Tool layer:** small deterministic functions for policy lookup, receipt validation, limit calculations, and approval thresholds.
- **Policy guardrail:** deterministic final decision logic prevents the LLM from overriding explicit rules.
- **Output validator:** asserts the exact required schema and allowed decision values.

This gives the assignment an actual agentic workflow without making correctness depend on probabilistic arithmetic.

### Manual Review choices

- `CLM-004`: Manual Review because it contains business-class airfare, a missing lodging receipt, and a post-cap amount above $2,000. The policy explicitly says business/first-class airfare is an exception and claims over $2,000 require Manual Review.
- `CLM-005`: Manual Review because the meal expense is above $25 and has no receipt. The missing-receipt rule explicitly says not to silently reject it.
- Claims with purely ineligible items are rejected rather than manually reviewed because the policy's decision guidance explicitly defines that case as `REJECT`.

### Assumptions

1. For timeliness, the trip start/expense date is used as the claim's expense date because the supplied sample claims do not provide a separate line-level expense date.
2. Meal/lodging caps are applied using the supplied `days`/`nights` information.
3. For Manual Review, `approved_amount` and `deducted_amount` are reported as `0.0` because no final payment decision is authorized yet. The explanation and policy references preserve the reasons for review.
4. Confidence describes confidence in the **classification**, not a probability that the employee will ultimately be paid.

### Trade-offs

- No vector database is needed because the assignment provides a tiny, stable policy. A structured policy dictionary is more transparent and easier to audit.
- No multi-agent framework is necessary. A small tool registry is easier to understand and satisfies the required agent/tool workflow.
- The optional LLM dependency is isolated so the notebook remains runnable without paid infrastructure.

### What I would improve next

- Add line-level receipt/document extraction from PDFs or images.
- Add duplicate-claim detection.
- Add a real policy document retriever with source spans.
- Add human-review feedback capture and evaluation metrics.
- Add persistent audit storage and authentication if this became a real internal application.
- Add automated regression tests for every policy rule and boundary value.


## 6. Final Results

The following cell prints the required result format for all five claims.

In [ ]:

# Required output.
FINAL_RESULTS = []
for r in all_results:
    FINAL_RESULTS.append({
        "claim_id": r["claim_id"],
        "decision": r["decision"],
        "approved_amount": r["approved_amount"],
        "deducted_amount": r["deducted_amount"],
        "missing_docs": r["missing_docs"],
        "policy_refs": r["policy_refs"],
        "confidence": r["confidence"],
        "explanation": r["explanation"],
        "tools_used": r["tools_used"],
    })

assert len(FINAL_RESULTS) == 5
for item in FINAL_RESULTS:
    assert list(item.keys()) == [
        "claim_id", "decision", "approved_amount", "deducted_amount",
        "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"
    ]

print(json.dumps(FINAL_RESULTS, indent=2))
